# Solutions of the audience exercises
Please try the exercises in `lecture.ipynb` first. Just press Runtime → Run all. The first cell downloads the repository.


In [ ]:

import sys, os, subprocess
if not os.path.exists("src"):                      # fresh Colab: get the repository first
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Umadin1827/robot-project.git"], check=True)
    os.chdir("robot-project")
sys.path.insert(0, os.getcwd())
import numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 14, "axes.grid": True})
from src.paths import spiral_path
from src.robot import RobotParams, simulate
from src.controllers import PurePursuit, wheel_speeds
from src.metrics import cross_track_error
p = RobotParams(); cx, cy = spiral_path(800); SEED = 0


## Exercise 1 (easy)
- `Ld = 0.8`: the robot cuts the corners of the loops, the RMS error gets much bigger and it needs more time (see the sweep in section 6.3).
- `Ld = 0.05`: without sensor noise the error is even smaller (below 1 mm). So in a perfect simulation small is better. The price is paid with a noisy sensor, see exercise 3.

## Exercise 2 (medium)

In [ ]:

def my_wheel_speeds(v, kappa, B):
    """Return (v_left, v_right) in m/s."""
    omega = v * kappa                    # turning rate
    v_left = v - omega * B / 2           # the mean of both wheels is v, the difference is B*omega
    v_right = v + omega * B / 2
    return v_left, v_right

ok = all(np.allclose(my_wheel_speeds(0.2, k, 0.15), wheel_speeds(0.2, k, 0.15)) for k in (0.0, 1.0, -2.0))
print("✅ correct!" if ok else "❌ not yet")


## Exercise 3 (stretch)

In [ ]:

ld_noisy = [0.1, 0.2, 0.3, 0.5, 0.8]
rms_noisy, chatter = [], []
for Ld_try in ld_noisy:
    r_try = simulate(PurePursuit(cx, cy, p, Ld=Ld_try, v_ref=0.25), p, t_max=200, noise_std=0.1, seed=SEED)
    rms_noisy.append(np.sqrt(np.mean(cross_track_error(r_try.x, r_try.y, cx, cy)**2)) * 100)
    chatter.append(np.std(np.diff(r_try.u_l)))              # how nervous is the left motor voltage?
fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))
axs[0].plot(ld_noisy, rms_noisy, "o-"); axs[0].set(xlabel="look-ahead distance $L_d$ [m]", ylabel="RMS error [cm]", title="Tracking error")
axs[1].plot(ld_noisy, chatter, "o-", color="tab:red"); axs[1].set(xlabel="look-ahead distance $L_d$ [m]", ylabel="voltage chatter [V]", title="Nervousness of the motor voltage")
plt.tight_layout(); plt.show()
best = ld_noisy[int(np.argmin(rms_noisy))]
print(f"smallest RMS error at Ld = {best} m")


**Discussion.** With a noisy sensor the RMS error is smallest at a *medium* $L_d$ (printed above), because a very small $L_d$ follows every noise kick and a big one cuts corners. The chatter of the voltage keeps falling when $L_d$ grows. On a real robot I would take a value around the minimum of the error curve, or a bit larger to protect the motors and gearbox. Another way: filter the position before using it (extension).